# Lab 04 — Probabilistic Models

**Solved notebook.** Every task is solved and explained: a *How we got there* cell before each function gives the
reasoning, and an *Answers* cell after each experiment answers the guide's questions with the numbers this notebook
produces. Try the tasks in `lab04_probabilistic_models.ipynb` first, then compare.

| Part | Task | twin | Time |
|:--|:--|:--|:-:|
| A | MLE and MAP with `jax.grad`: Gaussian, Beta prior, ridge as MAP | `Ridge`, `BernoulliNB` (`alpha`) | 20 min |
| B | Naive Bayes, **discrete** features: Bernoulli, Laplace smoothing | `BernoulliNB` | 15 min |
| C | Naive Bayes, **continuous** features: Gaussian; categorical (binned) | `GaussianNB`, `CategoricalNB` | 20 min |
| D | Naive Bayes for **regression**: binned target; continuous target | `GaussianNB`, `CategoricalNB`, `LinearRegression` | 35 min |
| E | Probabilities: log-loss, Brier, reliability; decisions with costs | `log_loss`, `brier_score_loss`, `calibration_curve` | 20 min |
| F | Challenge: Platt scaling with `jax.grad` | `LogisticRegression` | home |

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax.scipy.special import logsumexp
from lab04_utils import (
    apply_bins,
    classification_metrics,
    compare,
    fit_bins,
    fit_standardizer,
    load_housing,
    load_spam,
    plot_beta_posteriors,
    plot_cost_curve,
    plot_learning_curves,
    plot_pred_vs_actual,
    plot_reliability,
    regression_metrics,
    standardize,
    stratified_split,
    train,
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.naive_bayes import BernoulliNB, CategoricalNB, GaussianNB
from sklearn.preprocessing import KBinsDiscretizer

plt.rcParams["figure.figsize"] = (9, 4)
rng = np.random.default_rng(0)

## The data

* **Spam** (classification): `X_s` raw frequencies; `B_s` = "the word appears" (binary features for Bernoulli NB);
  `L_s` = $\log(1+x)$ (continuous features for Gaussian NB); `Z_s` = 4 quantile bins of `L_s` (categorical features).
* **House prices** (regression): standardized features, target in units of 100 000 USD.

Bin edges and scaling statistics are fitted on the **training** part only.

In [ ]:
X_s, y_s, names_s = load_spam()
X_h, y_h, names_h = load_housing()

tr_s, te_s = stratified_split(y_s, 0.2, rng)
ytr_s, yte_s = y_s[tr_s], y_s[te_s]
B_s, L_s = (X_s > 0).astype(float), np.log1p(X_s)
edges_s = fit_bins(L_s[tr_s], 4)
Z_s = apply_bins(L_s, edges_s)

perm = rng.permutation(len(y_h))
te_h, tr_h = perm[: len(y_h) // 5], perm[len(y_h) // 5 :]
mean_h, std_h = fit_standardizer(X_h[tr_h])
Xtr_h, Xte_h = standardize(X_h[tr_h], mean_h, std_h), standardize(X_h[te_h], mean_h, std_h)
ytr_h, yte_h = y_h[tr_h], y_h[te_h]
print("spam train/test:", len(tr_s), len(te_s), "| housing train/test:", len(tr_h), len(te_h))

## Part A — MLE and MAP with `jax.grad`

### A1. Gaussian maximum likelihood

Implement `gaussian_nll(params, x)`, the negative log-likelihood of $x_1,\dots,x_n\sim\mathcal{N}(\mu,\sigma^2)$ with
`params = {"mu": ..., "log_sigma": ...}` (optimizing $\log\sigma$ keeps $\sigma > 0$):

$$-\ell(\mu,\sigma) = \frac n2\log(2\pi\sigma^2) + \frac{1}{2\sigma^2}\sum_i (x_i-\mu)^2$$

### How we got there (A1)

We write the formula with `jnp` and `jnp.exp(2 * log_sigma)` for $\sigma^2$: nothing else. `train` differentiates it. The
maximum is where the gradient vanishes, which happens at $\hat\mu = \bar x$ and $\hat\sigma^2 = \frac1n\sum(x_i-\bar x)^2$
(divide by $n$: the *biased* estimate, since the likelihood measures the spread around the sample mean).

In [ ]:
def gaussian_nll(params, x) -> jax.Array:
    sigma2 = jnp.exp(2 * params["log_sigma"])
    return 0.5 * x.size * jnp.log(2 * jnp.pi * sigma2) + jnp.sum((x - params["mu"]) ** 2) / (2 * sigma2)

In [ ]:
x = jnp.array([2.1, 1.9, 2.6, 2.4, 3.0])
fit, _ = train(gaussian_nll, {"mu": jnp.array(0.0), "log_sigma": jnp.array(0.0)}, x, lr=0.05, steps=3000)
var_hat = float(jnp.exp(2 * fit["log_sigma"]))
compare("mu vs the sample mean", fit["mu"], x.mean())
compare("variance vs np.var (divide by n)", var_hat, np.var(np.asarray(x)))
print(f"variance by gradient {var_hat:.4f} | divide by n {np.var(np.asarray(x)):.4f} | divide by n-1 {np.var(np.asarray(x), ddof=1):.4f}")
sim = np.random.default_rng(1).normal(0, 1, (100_000, 5))
print(f"mean of the MLE variance of 100000 samples of size 5 from N(0, 1): {sim.var(axis=1).mean():.3f}")

### Answers (A1)

* Gradient descent on the negative log-likelihood finds $\hat\mu$ equal to the sample mean (difference $4\cdot10^{-16}$) and
  $\hat\sigma^2 = 0.1480$, equal to `np.var` **dividing by $n$** (difference $3\cdot10^{-17}$). Dividing by $n-1$ would give 0.1850.
* The MLE variance is **biased**: averaged over 100 000 samples of size 5 from $\mathcal{N}(0,1)$ it is 0.798, not 1, i.e. the
  factor $(n-1)/n = 0.8$. The likelihood measures the spread around the *sample* mean, which is closer to the data than
  the true mean. The bias disappears with $n$; Gaussian Naive Bayes uses the MLE variance (Part C).

### A2. MAP for a coin: the Beta prior

The log-posterior (up to a constant) of a coin with $k$ heads in $n$ tosses under a $\text{Beta}(a, b)$ prior is
$(k + a - 1)\log\theta + (n - k + b - 1)\log(1-\theta)$. Implement `log_posterior_bernoulli(theta, k, n, a, b)`. The
driver evaluates it on a grid of $\theta$ with `jax.vmap`, takes the maximum, and compares with the closed form
$\hat\theta_{MAP} = (k + a - 1)/(n + a + b - 2)$. Then it checks that `BernoulliNB(alpha)` uses exactly this estimate
with $a = b = \alpha + 1$.

### How we got there (A2)

The likelihood is $\theta^k(1-\theta)^{n-k}$ and the Beta prior contributes $\theta^{a-1}(1-\theta)^{b-1}$: the
exponents add, and the logarithm turns the product into the sum above. `jnp.log1p(-theta)` is $\log(1-\theta)$. `jax.vmap`
evaluates the same scalar function on the whole grid: no loop. The maximum of the grid agrees with the closed form to
the grid resolution.

In [ ]:
def log_posterior_bernoulli(theta, k, n, a, b) -> jax.Array:
    return (k + a - 1) * jnp.log(theta) + (n - k + b - 1) * jnp.log1p(-theta)

In [ ]:
grid = jnp.linspace(0.0005, 0.9995, 1999)
print(" n   k  prior      grid MAP   formula")
for n, k in ((3, 3), (10, 7), (30, 21)):
    for a, b in ((1, 1), (2, 2), (10, 10)):
        lp = jax.vmap(lambda t, k=k, n=n, a=a, b=b: log_posterior_bernoulli(t, k, n, a, b))(grid)
        print(f"{n:2d}  {k:2d}  Beta({a},{b}) {float(grid[jnp.argmax(lp)]):9.4f}  {(k + a - 1) / (n + a + b - 2):8.4f}")

plot_beta_posteriors(3, 3, {"flat": (1, 1), "Laplace": (2, 2), "strong": (10, 10)})
k_word, n_spam = 0, 8                                         # a word never seen in 8 spam e-mails
Xw = np.array([[1]] * k_word + [[0]] * (n_spam - k_word) + [[1]] * 8 + [[0]] * 2)
yw = np.array([1] * n_spam + [0] * 10)
for alpha in (1.0, 3.0):
    theta_nb = np.exp(BernoulliNB(alpha=alpha).fit(Xw, yw).feature_log_prob_[1, 0])
    print(f"BernoulliNB(alpha={alpha}): P(word | spam) = {theta_nb:.4f} ; MAP with Beta({int(alpha) + 1},{int(alpha) + 1}): "
          f"{(k_word + alpha) / (n_spam + 2 * alpha):.4f}")

### Answers (A2)

The grid maximum equals the closed form in every case (to the grid resolution 0.0005): e.g. after 3 heads in 3 tosses the MLE
(flat prior) is $\theta = 1$, the Beta(2,2) MAP is **0.800** and the strong Beta(10,10) MAP is **0.571**; after 30 tosses
(21 heads) they are 0.700, 0.688 and 0.625: the prior fades with data, and a strong prior needs more of it to be overcome.
`BernoulliNB(alpha)` is the MAP estimate under a Beta($\alpha+1$, $\alpha+1$) prior: for a word seen in 0 of 8 spam
e-mails, $\alpha = 1$ gives $\theta = 1/10 = 0.1000$ (Beta(2,2)) and $\alpha = 3$ gives $3/14 = 0.2143$ (Beta(4,4)), both equal to
scikit-learn's value. **Laplace smoothing is a Bayesian prior**: $\alpha$ imaginary observations of each outcome.

### A3. MAP for regression: ridge is a Gaussian prior

Linear regression with Gaussian noise $\mathcal{N}(0,\sigma^2)$ and a Gaussian prior $w_j\sim\mathcal{N}(0,\tau^2)$ (no prior
on the bias). Implement `map_loss(params, X, y, sigma2, tau2)`, the negative log-posterior without constants,
$\frac{1}{2\sigma^2}\sum_i (y_i - w^\top x_i - b)^2 + \frac{1}{2\tau^2}\lVert w\rVert^2$, and train it. Compare with
`Ridge(alpha = sigma2 / tau2)`.

### How we got there (A3)

Two terms: the data term is the Gaussian log-likelihood (a *sum* of squares divided by $2\sigma^2$), the prior term the
Gaussian log-prior. Multiplying the whole loss by $2\sigma^2$ gives $\sum r_i^2 + (\sigma^2/\tau^2)\lVert w\rVert^2$,
which is scikit-learn's ridge objective with $\alpha = \sigma^2/\tau^2$. The bias is left out of the penalty, as in
`Ridge`.

In [ ]:
def map_loss(params, X, y, sigma2, tau2) -> jax.Array:
    resid = y - (X @ params["w"] + params["b"])
    return jnp.sum(resid**2) / (2 * sigma2) + jnp.sum(params["w"] ** 2) / (2 * tau2)

In [ ]:
sub = np.random.default_rng(5).choice(len(ytr_h), 100, replace=False)
sigma2, tau2 = 0.4, 0.25
p, _ = train(map_loss, {"w": jnp.zeros(8), "b": jnp.zeros(())}, Xtr_h[sub], ytr_h[sub], sigma2, tau2, lr=0.05, steps=4000)
rr = Ridge(alpha=sigma2 / tau2).fit(Xtr_h[sub], ytr_h[sub])
compare("MAP weights vs Ridge", p["w"], rr.coef_)
compare("MAP bias vs Ridge", p["b"], rr.intercept_)
ols = LinearRegression().fit(Xtr_h[sub], ytr_h[sub])
print("norm of the weights: least squares", float(np.linalg.norm(ols.coef_)), "| MAP", float(jnp.linalg.norm(p["w"])))

### Answers (A3)

The MAP weights found by `jax.grad` equal `Ridge(alpha = sigma2 / tau2)` (here $\alpha = 0.4/0.25 = 1.6$) to $1.1\cdot10^{-15}$
and the bias to $8.9\cdot10^{-16}$: **ridge regression is the MAP estimate under Gaussian noise and a Gaussian prior on the weights**.
The prior shrinks the weights: their norm is 1.740 for least squares and 1.468 for the MAP. The ratio $\sigma^2/\tau^2$ is the
regularization strength: a tight prior (small $\tau^2$) or noisy data (large $\sigma^2$) means more shrinkage. In Class 03 the same
penalty was chosen because it *works*; here we see *why*: it is a prior belief that the weights are small.

## Part B — Naive Bayes with discrete features

$$\hat c = \arg\max_c\Big[\log P(c) + \sum_j \log P(x_j\mid c)\Big]$$

Training is **counting**. The functions take `n_classes` (2 for spam; Part D uses more).

### B1. Bernoulli Naive Bayes

Binary features $b_j = [x_j > 0]$. Implement `fit_bernoulli_nb(Xb, y, alpha, n_classes=2)` returning the log-prior of each
class and the logs of $\theta_{cj} = P(b_j=1\mid c) = (n_{cj}+\alpha)/(n_c+2\alpha)$ and of $1-\theta_{cj}$;
`joint_log_bernoulli(model, Xb)` returning $\log P(c) + \log P(x\mid c)$ for every example and class; and
`posterior_proba(joint)`, the normalized $P(c\mid x)$ computed with `logsumexp` (subtract the log-normalizer, then
`exp`).

### How we got there (B1)

For each class we count the examples (`(y == c).sum()`) and how often each word appears (`Xb[y == c].sum(axis=0)`); the
smoothed ratio is $\theta$. The score of a word is $\log\theta$ if present and $\log(1-\theta)$ if not:
`jnp.where(Xb == 1, log_theta, log_one_minus)`, broadcast to `(n, classes, d)` and summed over the features (the
product $b\log\theta + (1-b)\log(1-\theta)$ would compute $0\cdot(-\infty)$ for an unseen word). The posterior divides
by $P(x) = \sum_c P(c)P(x\mid c)$: in logs, subtract `logsumexp` over the classes, which is numerically safe.

In [ ]:
def fit_bernoulli_nb(Xb, y, alpha, n_classes=2) -> tuple[jax.Array, jax.Array, jax.Array]:
    log_prior = jnp.log(jnp.array([np.mean(y == c) for c in range(n_classes)]))
    theta = jnp.stack([(Xb[y == c].sum(axis=0) + alpha) / ((y == c).sum() + 2 * alpha) for c in range(n_classes)])
    return log_prior, jnp.log(theta), jnp.log1p(-theta)


def joint_log_bernoulli(model, Xb) -> jax.Array:
    log_prior, log_theta, log_one_minus = model
    per_feature = jnp.where(Xb[:, None, :] == 1, log_theta[None], log_one_minus[None])
    return log_prior + per_feature.sum(axis=-1)


def posterior_proba(joint) -> jax.Array:
    return jnp.exp(joint - logsumexp(joint, axis=1, keepdims=True))

In [ ]:
Btr, Bte = B_s[tr_s], B_s[te_s]
bnb = fit_bernoulli_nb(Btr, ytr_s, 1.0)
bnb_sk = BernoulliNB(alpha=1.0).fit(Btr, ytr_s)
joint_b = joint_log_bernoulli(bnb, Bte)
compare("joint log-probabilities", joint_b, bnb_sk.predict_joint_log_proba(Bte))
compare("posterior probabilities", posterior_proba(joint_b)[:, 1], bnb_sk.predict_proba(Bte)[:, 1])
pred_b = np.asarray(joint_b).argmax(axis=1)
print("Bernoulli NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_b).items()})

# Laplace smoothing: alpha = 0 makes a class impossible when a word was never seen in it.
for n in (20, 100):
    for alpha in (0.0, 1.0):
        with np.errstate(all="ignore"):
            j = np.asarray(joint_log_bernoulli(fit_bernoulli_nb(B_s[tr_s[:n]], ytr_s[:n], alpha), Bte))
        print(f"n = {n:3d}, alpha = {alpha}: e-mails with an impossible class {int(np.isneginf(j).any(axis=1).sum()):3d} of {len(Bte)}, accuracy {np.mean(j.argmax(axis=1) == yte_s):.3f}")

### Answers (B1)

* **Match.** Our joint log-probabilities differ from `BernoulliNB(alpha=1)` by $2.8\cdot10^{-14}$ and the posterior
  probabilities by $3.6\cdot10^{-15}$. Test set: accuracy 0.887, precision 0.880, recall 0.826, $F_1$ 0.852, MCC 0.762.
* **Laplace smoothing.** With $\alpha = 0$ and 20 training e-mails, **771 of the 921** test e-mails contain a word never seen in
  one class, so that class gets $-\infty$ (accuracy 0.788, against 0.806 with $\alpha = 1$). With 100 training e-mails the
  numbers are 555 impossible and accuracy 0.864 against 0.887. The problem fades as counts grow but never for **rare**
  words: a zero count must not become a zero probability.

## Part C — Naive Bayes with continuous features

### C1. Gaussian Naive Bayes

Implement `fit_gaussian_nb(X, y, var_smoothing=1e-9, n_classes=2)` returning the log-prior, the class means and the class
variances (the maximum-likelihood variance, **dividing by $n$**; add `var_smoothing * max_j Var(x_j)` to every variance),
and `joint_log_gaussian(model, X)`. Broadcast the examples `(n, 1, d)` against the class parameters `(1, K, d)`: no loops
over examples. Use the log-transformed features `L_s`.

### How we got there (C1)

The maximum-likelihood estimates of a Gaussian are the class mean and the *biased* variance (`X.var()`), exactly A1.
The log-density of a feature is $-\frac12[\log(2\pi\sigma^2) + (x-\mu)^2/\sigma^2]$, summed over the features.
Broadcasting evaluates all examples and classes at once. The variance floor plays the role of Laplace smoothing: a feature
that is constant inside a class would otherwise have $\sigma^2=0$.

In [ ]:
def fit_gaussian_nb(X, y, var_smoothing=1e-9, n_classes=2) -> tuple[jax.Array, jax.Array, jax.Array]:
    floor = var_smoothing * X.var(axis=0).max()
    mean = jnp.stack([X[y == c].mean(axis=0) for c in range(n_classes)])
    var = jnp.stack([X[y == c].var(axis=0) for c in range(n_classes)]) + floor
    return jnp.log(jnp.array([np.mean(y == c) for c in range(n_classes)])), mean, var


def joint_log_gaussian(model, X) -> jax.Array:
    log_prior, mean, var = model
    per_feature = jnp.log(2 * jnp.pi * var)[None] + (X[:, None, :] - mean[None]) ** 2 / var[None]
    return log_prior - 0.5 * per_feature.sum(axis=-1)

In [ ]:
gnb = fit_gaussian_nb(L_s[tr_s], ytr_s)
gnb_sk = GaussianNB().fit(L_s[tr_s], ytr_s)
compare("class means", gnb[1], gnb_sk.theta_)
compare("class variances", gnb[2], gnb_sk.var_)
joint_g = joint_log_gaussian(gnb, L_s[te_s])
compare("joint log-probabilities", joint_g, gnb_sk.predict_joint_log_proba(L_s[te_s]))
pred_g = np.asarray(joint_g).argmax(axis=1)
print("Gaussian NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_g).items()})

### Answers (C1)

Class means and variances equal `GaussianNB`'s exactly (difference 0), the joint log-probabilities to $2.9\cdot10^{-11}$. Test
set: accuracy 0.853, precision 0.747, **recall 0.950**, $F_1$ 0.836, MCC 0.724. High recall and low precision: it flags almost
every spam, but also many good e-mails. The features are far from independent given the class and far from Gaussian (most
frequencies are exactly 0, even after the logarithm); the product of many correlated Gaussian factors is over-confident,
so many e-mails end up on the spam side.

### C2. Categorical Naive Bayes (discretized continuous features)

Instead of a Gaussian, cut each feature into bins and treat the bin as a category with its own probability. Implement
`fit_categorical_nb(Z, y, alpha, n_categories, n_classes=2)` (`Z` holds integer bins) returning the log-prior and
$\log\theta_{cjm} = \log\frac{n_{cjm}+\alpha}{n_{cj}+\alpha\,M}$ for class $c$, feature $j$, category $m$
($M$ = `n_categories`), and `joint_log_categorical(model, Z)`.

### How we got there (C2)

`np.bincount(Z[y == c, j], minlength=M)` counts how often each category occurs for feature $j$ within class $c$: a table
of shape `(classes, d, M)`. The smoothed probability divides by the total count plus $\alpha M$. To score an example we
pick, for each feature, the log-probability of its own category: the fancy index `log_theta[:, j, Z]` does it for all
examples at once, giving `(classes, n, d)`, which we sum over the features. It is the Bernoulli model with $M$ outcomes
instead of 2.

In [ ]:
def fit_categorical_nb(Z, y, alpha, n_categories, n_classes=2) -> tuple[jax.Array, jax.Array]:
    log_prior = jnp.log(jnp.array([np.mean(y == c) for c in range(n_classes)]))
    counts = np.stack([[np.bincount(Z[y == c, j], minlength=n_categories) for j in range(Z.shape[1])] for c in range(n_classes)])
    theta = (counts + alpha) / (counts.sum(axis=-1, keepdims=True) + alpha * n_categories)
    return log_prior, jnp.log(theta)


def joint_log_categorical(model, Z) -> jax.Array:
    log_prior, log_theta = model
    picked = log_theta[:, jnp.arange(Z.shape[1])[None, :], Z]
    return log_prior + picked.sum(axis=-1).T

In [ ]:
cnb = fit_categorical_nb(Z_s[tr_s], ytr_s, 1.0, 4)
cnb_sk = CategoricalNB(alpha=1.0, min_categories=4).fit(Z_s[tr_s], ytr_s)
joint_c = joint_log_categorical(cnb, Z_s[te_s])
compare("joint log-probabilities", joint_c, cnb_sk.predict_joint_log_proba(Z_s[te_s]))
pred_c = np.asarray(joint_c).argmax(axis=1)
print("Categorical NB", {k: round(v, 3) for k, v in classification_metrics(yte_s, pred_c).items()})

# Generative or discriminative? Test accuracy against the number of training e-mails.
sizes = [20, 50, 100, 300, 1000, len(tr_s)]
curves = {"logistic": [], "Bernoulli NB": [], "Gaussian NB": [], "Categorical NB": []}
for n in sizes:
    idx = tr_s[:n]
    m_, s_ = fit_standardizer(L_s[idx])
    lr_ = LogisticRegression(C=1.0, max_iter=5000).fit(standardize(L_s[idx], m_, s_), y_s[idx])
    curves["logistic"].append(lr_.score(standardize(L_s[te_s], m_, s_), yte_s))
    curves["Bernoulli NB"].append(np.mean(np.asarray(joint_log_bernoulli(fit_bernoulli_nb(B_s[idx], y_s[idx], 1.0), Bte)).argmax(1) == yte_s))
    curves["Gaussian NB"].append(np.mean(np.asarray(joint_log_gaussian(fit_gaussian_nb(L_s[idx], y_s[idx]), L_s[te_s])).argmax(1) == yte_s))
    curves["Categorical NB"].append(np.mean(np.asarray(joint_log_categorical(fit_categorical_nb(Z_s[idx], y_s[idx], 1.0, 4), Z_s[te_s])).argmax(1) == yte_s))
plot_learning_curves(sizes, curves)
print({k: [round(float(v), 3) for v in vs] for k, vs in curves.items()})

### Answers (C2)

* **Categorical NB.** Our joint log-probabilities equal `CategoricalNB`'s to $2.1\cdot10^{-14}$. With 4 quantile bins per feature:
  accuracy 0.852, precision 0.811, recall 0.815, $F_1$ 0.813, MCC 0.691. It is more balanced than the Gaussian model (no shape is
  assumed), but many word features have all their mass at 0, so their bins collapse and carry no more information than
  the Bernoulli "present / absent" (accuracy 0.887 is still the best Naive Bayes on spam).
* **Generative or discriminative.** Test accuracy for 20, 50, 100, 300, 1000 and 3680 training e-mails: logistic 0.730,
  0.862, 0.878, 0.898, 0.936, 0.935; Bernoulli NB 0.806, 0.862, 0.887, 0.877, 0.888, 0.887; Gaussian NB 0.831, 0.819, 0.819, 0.790,
  0.845, 0.853; Categorical NB 0.777, 0.822, 0.847, 0.847, 0.856, 0.852. With **20 examples** the Naive Bayes models beat
  logistic regression (few parameters, a strong assumption); from about 300 examples logistic regression is ahead and keeps
  improving, while the NB curves plateau at the error of the independence assumption. (One subset per size: the curves
  are noisy; the demo averages 10 subsets.)

## Part D — Naive Bayes for regression

The target is now a number. Two routes.

### D1. Discretize the target

Cut $y$ into $K$ quantile bins. Implement `bin_target(y, n_bins)` returning the interior **edges** (the quantiles at
$1/K, 2/K, \dots$: `np.quantile`) and the **bin means** $m_k$ (the mean of the training targets in each bin; the class
of a value is `np.searchsorted(edges, y, side="right")`), and `predict_expected(joint, means)`, the expected value
$\hat y = \sum_k P(k\mid x)\,m_k$ from the joint log-scores of a classifier (use `posterior_proba`).

The driver trains your Gaussian NB (Part C) on the bin labels, predicts the expected value and the most probable bin, and
compares with scikit-learn's `KBinsDiscretizer` + `GaussianNB` version.

### How we got there (D1)

The bin edges are the training quantiles; a value belongs to the bin whose edges bracket it (`searchsorted` on the
interior edges). The bin mean is the natural representative value. A classifier gives the *posterior over bins*; the
expected value is its dot product with the bin means. The most probable bin throws away everything except one number
per bin, which is why the expectation is better.

In [ ]:
def bin_target(y, n_bins) -> tuple[np.ndarray, np.ndarray]:
    edges = np.quantile(y, np.linspace(0, 1, n_bins + 1)[1:-1])
    classes = np.searchsorted(edges, y, side="right")
    means = np.array([y[classes == k].mean() for k in range(n_bins)])
    return edges, means


def predict_expected(joint, means) -> np.ndarray:
    return np.asarray(posterior_proba(joint)) @ means

In [ ]:
def report(name, pred):
    m = regression_metrics(yte_h, pred)
    print(f"{name:<40} R2 {m['r2']:.3f}  RMSE {m['rmse']:.3f}  MAE {m['mae']:.3f}")


report("mean baseline", np.full(len(yte_h), ytr_h.mean()))
lin_pred = LinearRegression().fit(Xtr_h, ytr_h).predict(Xte_h)
report("linear regression (Class 03)", lin_pred)

K = 10
edges_y, means_y = bin_target(ytr_h, K)
cls_h = np.searchsorted(edges_y, ytr_h, side="right")
joint_h = joint_log_gaussian(fit_gaussian_nb(Xtr_h, cls_h, n_classes=K), Xte_h)
pred_exp = predict_expected(joint_h, means_y)
pred_arg = means_y[np.asarray(joint_h).argmax(axis=1)]
report(f"Gaussian NB, K = {K}: expected value", pred_exp)
report(f"Gaussian NB, K = {K}: most probable bin", pred_arg)

kb = KBinsDiscretizer(K, encode="ordinal", strategy="quantile").fit(ytr_h[:, None])
cls_sk = np.asarray(kb.transform(ytr_h[:, None])).ravel().astype(int)
nb_sk = GaussianNB().fit(Xtr_h, cls_sk)
compare("bin labels vs KBinsDiscretizer", cls_h, cls_sk)
compare("expected value vs scikit-learn", pred_exp, nb_sk.predict_proba(Xte_h) @ means_y[nb_sk.classes_])
for K_ in (5, 20):
    e_, m_ = bin_target(ytr_h, K_)
    j_ = joint_log_gaussian(fit_gaussian_nb(Xtr_h, np.searchsorted(e_, ytr_h, side="right"), n_classes=K_), Xte_h)
    report(f"Gaussian NB, K = {K_}: expected value", predict_expected(j_, m_))

### Answers (D1)

* **Match.** The bin labels equal `KBinsDiscretizer`'s (difference 0) and the expected values equal the scikit-learn
  computation to $1.6\cdot10^{-14}$.
* **Results** (housing test set; mean baseline $R^2 = 0$, RMSE 1.148; linear regression $R^2 = 0.674$, RMSE 0.655):

  | Gaussian NB on the binned target | $R^2$ | RMSE |
  |:--|--:|--:|
  | $K = 5$, expected value | 0.530 | 0.787 |
  | $K = 10$, expected value | **0.543** | 0.776 |
  | $K = 20$, expected value | 0.548 | 0.771 |
  | $K = 10$, most probable bin | 0.417 | 0.876 |

  Using the **whole posterior** (the expectation) is much better than the most probable bin, which returns one of only
  $K$ values. More bins help a little (the target is better resolved). Every version is below linear
  regression: the features of this data are correlated and Naive Bayes ignores that.
* A bonus that linear regression does not have: a full **distribution over prices** for every district.

### D2. Discrete features for regression

Now discretize the **features** too (5 quantile bins each, `fit_bins` / `apply_bins`) and use your categorical NB with
the $K$ target bins. The driver reports the result for 3, 5, 10 and 20 bins per feature and compares with
`CategoricalNB`.

In [ ]:
def cat_reg(bins, K=10):
    edges_x = fit_bins(Xtr_h, bins)
    Ftr, Fte = apply_bins(Xtr_h, edges_x), apply_bins(Xte_h, edges_x)
    e_, m_ = bin_target(ytr_h, K)
    cls = np.searchsorted(e_, ytr_h, side="right")
    joint = joint_log_categorical(fit_categorical_nb(Ftr, cls, 1.0, bins, n_classes=K), Fte)
    sk = CategoricalNB(alpha=1.0, min_categories=bins).fit(Ftr, cls)
    return predict_expected(joint, m_), sk.predict_proba(Fte) @ m_[sk.classes_]


for bins in (3, 5, 10, 20):
    ours, theirs = cat_reg(bins)
    report(f"Categorical NB, K = 10, {bins:2d} bins per feature", ours)
    compare(f"   vs CategoricalNB ({bins} bins)", ours, theirs)

### Answers (D2)

The categorical model equals `CategoricalNB` to $10^{-14}$ at every setting. Test $R^2$ against the bins per feature: 3 bins
**0.389**, 5 bins 0.541, 10 bins 0.610, **20 bins 0.651**. Coarse bins lose the shape of each feature; with 20 bins the
counts approximate each $P(x_j\mid k)$ non-parametrically: it beats the Gaussian version (0.543) and approaches linear
regression (0.674). With enough data per cell, discretizing is a strong option; with too many bins the cells become
too small and the smoothing $\alpha$ dominates.

### D3. A continuous target: linear-Gaussian Naive Bayes

$$y\sim\mathcal{N}(\mu_y,\tau^2),\qquad x_j\mid y\sim\mathcal{N}(a_j + b_j\,y,\;s_j^2)\ \text{(independent given } y)$$

Implement `fit_lgnb(X, y)` returning $(\mu_y, \tau^2, a, b, s^2)$: the mean and variance of $y$, the least-squares fit of every
feature **on** $y$ ($x_j\approx a_j+b_jy$; `np.linalg.lstsq` on `[y, 1]`), and the residual variances (mean squared
residual). Then `predict_lgnb(model, X)` returning the posterior mean and standard deviation of $y$:

$$P = \frac1{\tau^2} + \sum_j \frac{b_j^2}{s_j^2}, \qquad \hat y = \frac1P\Big(\frac{\mu_y}{\tau^2} + \sum_j\frac{b_j\,(x_j-a_j)}{s_j^2}\Big),
\qquad \text{sd} = \frac1{\sqrt P}$$

The driver **verifies** your formula without it: `neg_log_posterior(y, x, model)` is written in `jax.numpy`, `jax.grad`
climbs it for a few districts, and the maximum must equal your posterior mean.

### How we got there (D3)

By Bayes, $\log p(y\mid x) = \log p(y) + \sum_j\log p(x_j\mid y) + \text{const}$. Every term is a quadratic in $y$, so
the posterior is Gaussian, and completing the square gives the precision $P$ (the sum of the *coefficients* of $y^2$) and
the mean (the linear coefficient divided by $P$). The maximum-likelihood fit of $x_j\mid y$ is a *regression of the feature
on the target*: `np.linalg.lstsq(np.c_[y, 1], X)` fits all features at once and returns slopes $b_j$ and intercepts $a_j$.
Each feature adds precision $b_j^2/s_j^2$: an informative feature (large slope, small noise) has a heavy vote.

In [ ]:
def fit_lgnb(X, y) -> tuple[float, float, np.ndarray, np.ndarray, np.ndarray]:
    A = np.c_[y, np.ones(len(y))]
    coef = np.linalg.lstsq(A, X, rcond=None)[0]
    s2 = ((X - A @ coef) ** 2).mean(axis=0)
    return float(y.mean()), float(y.var()), coef[1], coef[0], s2


def predict_lgnb(model, X) -> tuple[np.ndarray, float]:
    mu, tau2, a, b, s2 = model
    precision = 1 / tau2 + np.sum(b**2 / s2)
    mean = (mu / tau2 + ((X - a) * b / s2).sum(axis=1)) / precision
    return mean, float(np.sqrt(1 / precision))

In [ ]:
model = fit_lgnb(Xtr_h, ytr_h)
mean_lg, sd_lg = predict_lgnb(model, Xte_h)
report("linear-Gaussian NB (continuous target)", mean_lg)
mu_y, tau2_y, a_j, b_j, s2_j = (jnp.asarray(v) for v in model)


def neg_log_posterior(y_, x):
    return 0.5 * (y_ - mu_y) ** 2 / tau2_y + 0.5 * jnp.sum((x - a_j - b_j * y_) ** 2 / s2_j)


grad_lp = jax.jit(jax.grad(neg_log_posterior))
climbed = []
for i in range(5):
    y_ = jnp.asarray(0.0)
    for _ in range(400):
        y_ = y_ - 0.1 * grad_lp(y_, jnp.asarray(Xte_h[i]))
    climbed.append(float(y_))
compare("posterior mean vs jax.grad ascent", mean_lg[:5], climbed)
inside = np.abs(yte_h - mean_lg) < 1.645 * sd_lg
print(f"posterior sd {sd_lg:.3f} (same for every district); the 90% interval covers {inside.mean():.3f} of the test districts")
plot_pred_vs_actual(yte_h, {"linear regression": lin_pred, "Gaussian NB, K = 10": pred_exp, "linear-Gaussian NB": mean_lg})

### Answers (D3)

* **The formula is right:** the posterior mean from your closed form equals the maximum of the log-posterior found by
  `jax.grad` ascent for five districts, to $3.6\cdot10^{-15}$.
* **Results:** $R^2 = 0.523$, RMSE 0.793 (linear regression 0.674, Gaussian NB on the binned target 0.543). One feature
  dominates the precision $\sum_j b_j^2/s_j^2$ (median income; the others add very little), and each feature is
  regressed on $y$ separately, so the correlations between the features are ignored.
* **Uncertainty:** the posterior standard deviation is 0.805 for **every** district (the precision does not depend on $x$),
  so the 90% interval $\hat y\pm1.645\,\text{sd}$ has the same width everywhere and covers **0.922** of the test districts, close to
  the nominal 0.90. The Gaussian-on-bins model of D1 gives a different distribution for every district.

### D4. Where the naive assumption hurts

Append $0, 1, 3, 10$ exact copies of the most informative feature (median income) to the features and refit **linear
regression** and your **linear-Gaussian NB**. The driver prints $R^2$, the posterior standard deviation and the coverage of
the 90% interval.

In [ ]:
j_inc = names_h.index("MedInc")
print("copies | R2 linear regression | R2 linear-Gaussian NB | posterior sd | 90% coverage")
for copies in (0, 1, 3, 10):
    Xa_tr = np.c_[Xtr_h, np.repeat(Xtr_h[:, [j_inc]], copies, axis=1)] if copies else Xtr_h
    Xa_te = np.c_[Xte_h, np.repeat(Xte_h[:, [j_inc]], copies, axis=1)] if copies else Xte_h
    r2_lin = regression_metrics(yte_h, LinearRegression().fit(Xa_tr, ytr_h).predict(Xa_te))["r2"]
    mean_, sd_ = predict_lgnb(fit_lgnb(Xa_tr, ytr_h), Xa_te)
    print(f"{copies:6d} | {r2_lin:20.3f} | {regression_metrics(yte_h, mean_)['r2']:21.3f} | {sd_:12.3f} | {np.mean(np.abs(yte_h - mean_) < 1.645 * sd_):.3f}")

### Answers (D4)

| copies of `MedInc` | $R^2$ linear regression | $R^2$ linear-Gaussian NB | posterior sd | 90% coverage |
|:-:|--:|--:|--:|--:|
| 0 | 0.674 | 0.523 | 0.805 | 0.922 |
| 1 | 0.674 | 0.453 | 0.673 | 0.855 |
| 3 | 0.674 | 0.303 | 0.531 | 0.694 |
| 10 | 0.674 | **0.085** | 0.349 | **0.464** |

Linear regression does not change (the copies share one weight). Naive Bayes adds the income evidence **again** for every
copy: the precision grows (the posterior sd falls from 0.805 to 0.349) while the prediction gets *worse*, so with 10 copies
the "90%" interval covers only 46% of the districts: the model is over-confident and wrong. Correlated features that
are not exact copies (rooms, bedrooms, occupancy) do the same in a milder form. This is the price of the naive assumption.

## Part E — Probabilities and decisions

A classifier's probability is only useful if it is **calibrated**. The driver trains a logistic regression on the standardized
log features (a discriminative model) to compare with your Naive Bayes.

### E1. Log-loss, Brier score and reliability

Implement `log_loss_(y, p)` $= -\frac1n\sum[y\log p + (1-y)\log(1-p)]$ (clip $p$ to $[10^{-15}, 1-10^{-15}]$),
`brier_score(y, p)` $= \frac1n\sum(p-y)^2$ and `reliability_table(y, p, bins)`, which cuts $[0,1]$ into `bins` equal bins and
returns, for every **non-empty** bin, the mean predicted probability, the observed frequency of spam and the count.
The driver checks them against `log_loss`, `brier_score_loss` and `calibration_curve`.

### How we got there (E1)

Log-loss is the mean negative log-likelihood of the labels (A2's likelihood, for a single Bernoulli per e-mail). The Brier
score is the mean squared error of the probability. A reliability table groups the e-mails by predicted probability
(`np.digitize` on the interior edges) and compares the average prediction with the observed rate: for a calibrated model
the two columns coincide.

In [ ]:
def log_loss_(y, p) -> float:
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def brier_score(y, p) -> float:
    return float(np.mean((p - y) ** 2))


def reliability_table(y, p, bins) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    which = np.digitize(p, np.linspace(0, 1, bins + 1)[1:-1])
    used = [b for b in range(bins) if (which == b).any()]
    return (np.array([p[which == b].mean() for b in used]), np.array([y[which == b].mean() for b in used]),
            np.array([(which == b).sum() for b in used]))

In [ ]:
Str_s = standardize(L_s[tr_s], *fit_standardizer(L_s[tr_s]))
Ste_s = standardize(L_s[te_s], *fit_standardizer(L_s[tr_s]))
lr = LogisticRegression(C=1.0, max_iter=5000).fit(Str_s, ytr_s)
probas = {
    "logistic": lr.predict_proba(Ste_s)[:, 1],
    "Bernoulli NB": np.asarray(posterior_proba(joint_b))[:, 1],
    "Gaussian NB": np.asarray(posterior_proba(joint_g))[:, 1],
}
for name, p in probas.items():
    print(f"{name:<13} accuracy {np.mean((p >= 0.5) == yte_s):.3f} | log-loss {log_loss_(yte_s, p):.3f} | Brier {brier_score(yte_s, p):.3f}")
p_ = probas["Gaussian NB"]
compare("log-loss vs sklearn", log_loss_(yte_s, p_), log_loss(yte_s, np.clip(p_, 1e-15, 1 - 1e-15)))
compare("Brier vs sklearn", brier_score(yte_s, p_), brier_score_loss(yte_s, p_))
frac, mean_pred = calibration_curve(yte_s, p_, n_bins=10, strategy="uniform")
mp, fr, cnt = reliability_table(yte_s, p_, 10)
compare("reliability: observed", fr, frac)
compare("reliability: predicted", mp, mean_pred)
plot_reliability(yte_s, probas)

### Answers (E1)

Our log-loss and Brier score equal scikit-learn's (difference 0) and the reliability table equals `calibration_curve`'s
(difference $10^{-19}$).

| model | accuracy | log-loss | Brier |
|:--|--:|--:|--:|
| logistic regression | 0.935 | **0.185** | **0.051** |
| Bernoulli NB | 0.887 | 0.615 | 0.100 |
| Gaussian NB | 0.853 | 3.141 | 0.145 |

Gaussian NB is only slightly less accurate than Bernoulli NB but its log-loss is five times larger: it makes **confident
mistakes**, which log-loss punishes without limit ($-\log 10^{-15} = 34$ for one bad e-mail) and the Brier score bounds by 1. The
reliability diagrams show the same: logistic regression lies near the diagonal, Gaussian NB puts almost everything at 0
or 1, yet the e-mails it calls "certain spam" are not all spam. Accuracy alone cannot see any of this.

### E2. Decisions with costs

A false positive (a real e-mail hidden) costs $c_{FP}$, a false negative (a spam delivered) costs $c_{FN}$. Implement
`bayes_threshold(c_fp, c_fn)` $= c_{FP}/(c_{FP}+c_{FN})$ (flag spam when $P(\text{spam}) >$ it) and
`expected_cost(y, p, threshold, c_fp, c_fn)`, the mean cost per e-mail. The driver compares, for the costs 1:1, 10:1 and 1:5,
the threshold 0.5, the Bayes threshold and the best threshold found by scanning the test set, for the logistic model.

### How we got there (E2)

Flagging costs $(1-p)\,c_{FP}$ in expectation (we are wrong if the e-mail is ham), letting it through costs $p\,c_{FN}$.
We flag when the first is smaller: $(1-p)\,c_{FP} < p\,c_{FN}$, i.e. $p > c_{FP}/(c_{FP}+c_{FN})$. The mean cost of a rule
is $(c_{FP}\cdot FP + c_{FN}\cdot FN)/n$, counting the two mistakes of the confusion matrix.

In [ ]:
def bayes_threshold(c_fp, c_fn) -> float:
    return c_fp / (c_fp + c_fn)


def expected_cost(y, p, threshold, c_fp, c_fn) -> float:
    flagged = p >= threshold
    return float((c_fp * np.sum(flagged & (y == 0)) + c_fn * np.sum(~flagged & (y == 1))) / len(y))

In [ ]:
p_lr = probas["logistic"]
grid_t = np.linspace(0.02, 0.99, 98)
for c_fp, c_fn in ((1, 1), (10, 1), (1, 5)):
    t_star = bayes_threshold(c_fp, c_fn)
    scan = np.array([expected_cost(yte_s, p_lr, t, c_fp, c_fn) for t in grid_t])
    print(f"FP:FN = {c_fp}:{c_fn}  t* = {t_star:.3f} | cost at 0.5: {expected_cost(yte_s, p_lr, 0.5, c_fp, c_fn):.3f} | cost at t*: "
          f"{expected_cost(yte_s, p_lr, t_star, c_fp, c_fn):.3f} | best scanned t = {grid_t[scan.argmin()]:.2f} (cost {scan.min():.3f})")
plot_cost_curve(grid_t, np.array([expected_cost(yte_s, p_lr, t, 10, 1) for t in grid_t]), bayes_threshold(10, 1))
# Naive Bayes probabilities are over-confident: the threshold computed from costs is wrong for them.
p_g = probas["Gaussian NB"]
print("Gaussian NB, costs 10:1: cost at t* =", round(expected_cost(yte_s, p_g, bayes_threshold(10, 1), 10, 1), 3),
      "| best scanned:", round(min(expected_cost(yte_s, p_g, t, 10, 1) for t in grid_t), 3))

### Answers (E2)

Expected cost per e-mail for the logistic model (mean of the costs):

| costs FP : FN | Bayes threshold $t^\star$ | cost at 0.5 | cost at $t^\star$ | best scanned threshold (cost) |
|:-:|:-:|--:|--:|:--|
| 1 : 1 | 0.500 | 0.065 | 0.065 | 0.44 (0.062) |
| 10 : 1 | 0.909 | 0.319 | **0.229** | 0.77 (0.203) |
| 1 : 5 | 0.167 | 0.213 | **0.166** | 0.14 (0.144) |

Moving from 0.5 to the cost-based threshold cuts the expected cost by 28% (10:1) and 22% (1:5), with no change to the
model. The theoretical threshold is *near* the best one found by scanning the test set, not equal to it: the test set
is finite, the probabilities are not perfectly calibrated, and the cost curve is flat near its minimum (0.229 at 0.909
against 0.203 at 0.77). And scanning the test set is exactly what we must **not** do; the cost-based $t^\star$ needs no
tuning. For **Gaussian NB** the cost at $t^\star$ is 1.257, five times the logistic model's: a threshold computed from
costs is only right for *calibrated* probabilities.

## Part F — Challenge: Platt scaling with `jax.grad`

Platt scaling learns a logistic map $p = \sigma(a\,t + b)$ from a monotone transform $t$ of the classifier's *score* $s$
(its log-odds) on a **calibration set** held out from the training data. The Naive Bayes log-odds are enormous
(they reach $\pm10^{8}$), so we first compress them: $t = \text{sign}(s)\log(1+|s|)$.

In [ ]:
def platt_loss(params, t, y) -> jax.Array:
    z = params["a"] * t + params["b"]
    return jnp.mean(jnp.logaddexp(0.0, z) - y * z)


n_fit = int(0.75 * len(tr_s))
fit_idx, cal_idx = tr_s[:n_fit], tr_s[n_fit:]
model_f = fit_gaussian_nb(L_s[fit_idx], y_s[fit_idx])


def score(X_) -> np.ndarray:
    j = np.asarray(joint_log_gaussian(model_f, X_))
    return j[:, 1] - j[:, 0]


s_cal, s_te = score(L_s[cal_idx]), score(L_s[te_s])
print(f"raw log-odds on the test set: from {s_te.min():.1e} to {s_te.max():.1e}, median |s| = {np.median(np.abs(s_te)):.0f}")


def compress(s) -> np.ndarray:
    return np.sign(s) * np.log1p(np.abs(s))                   # monotone: the ranking of the e-mails is unchanged


t_cal, t_te = compress(s_cal), compress(s_te)
platt, _ = train(platt_loss, {"a": jnp.array(1.0), "b": jnp.array(0.0)}, jnp.asarray(t_cal), jnp.asarray(y_s[cal_idx], dtype=float), lr=0.1, steps=3000)
twin = LogisticRegression(C=1e10, max_iter=10_000).fit(t_cal[:, None], y_s[cal_idx])
compare("Platt (a, b) vs LogisticRegression", [float(platt["a"]), float(platt["b"])], [twin.coef_[0, 0], twin.intercept_[0]])
print(f"a = {float(platt['a']):.4f}, b = {float(platt['b']):.4f}")

p_raw = np.asarray(jax.nn.sigmoid(jnp.asarray(s_te)))
p_cal = np.asarray(jax.nn.sigmoid(float(platt["a"]) * jnp.asarray(t_te) + float(platt["b"])))
t10 = bayes_threshold(10, 1)
print(f"{'':<22} log-loss  Brier  ECE  accuracy  cost (10:1) at t*")
for name, p in (("Gaussian NB, raw", p_raw), ("Gaussian NB + Platt", p_cal)):
    mp, fr, cnt = reliability_table(yte_s, p, 10)
    print(f"{name:<22} {log_loss_(yte_s, p):8.3f} {brier_score(yte_s, p):6.3f} {np.sum(cnt / cnt.sum() * np.abs(mp - fr)):5.3f} {np.mean((p >= 0.5) == yte_s):9.3f} {expected_cost(yte_s, p, t10, 10, 1):17.3f}")
plot_reliability(yte_s, {"raw": p_raw, "Platt scaled": p_cal})

### Answers (F)

* The raw log-odds of Gaussian NB reach $-6.7\cdot10^{8}$ (median $|s| = 116$): a symptom of over-confidence. After the monotone
  compression $t = \text{sign}(s)\log(1+|s|)$, Platt scaling by `jax.grad` gives $a = 0.4435$, $b = -1.1077$, equal to
  `LogisticRegression`'s to $1.3\cdot10^{-4}$.
* **Before and after** (test set):

  | | log-loss | Brier | ECE | accuracy | cost (10:1) at $t^\star$ |
  |:--|--:|--:|--:|--:|--:|
  | Gaussian NB, raw | 3.706 | 0.147 | 0.148 | 0.852 | 1.300 |
  | Gaussian NB + Platt | **0.312** | **0.095** | 0.107 | 0.868 | **0.390** |

  The log-loss falls by a factor 12 and the cost of the cost-based rule by 70%. The ranking of the e-mails does not
  change (the map is monotone); the accuracy changes a little because the intercept $b$ moves the 0.5 threshold. The ECE
  improves less than the log-loss: two parameters cannot straighten every bin (isotonic regression, used in the demo, is
  more flexible), and the compression is a heuristic.
* Calibration repaired the *probabilities*, not the model: it still ignores the correlations between features. A
  discriminative model such as logistic regression (log-loss 0.185) needs no repair.